# PostgreSQL queries with psycopg2

Use this notebook instead of a SQL client GUI to explore the database.

Uses the shared `scripts.db` helpers (same connection as `make seed` / `seed.ipynb`).

Run `make compose_up` and seed via `seed.ipynb` or `make seed`, then open this notebook at `http://localhost:8888`.

In [ ]:
from scripts.db import query

## Users

In [ ]:
query("SELECT id, username FROM users ORDER BY id LIMIT 10")

## Customers

In [ ]:
query("SELECT id, email, created_at FROM customers ORDER BY id LIMIT 10")

In [ ]:
query(
    "INSERT INTO customers (email) VALUES (%s) RETURNING id, email, created_at",
    ("notebook@example.com",),
)

## Orders

In [ ]:
query(
    "SELECT id, customer_id, status, created_at FROM orders ORDER BY id LIMIT 10"
)

In [ ]:
query(
    """
    SELECT id, customer_id, status, created_at
    FROM orders
    WHERE status = %s
    ORDER BY id
    LIMIT 10
    """,
    ("pending",),
)

## Order items

In [ ]:
query(
    """
    SELECT id, order_id, product_sku, quantity, unit_price_cents
    FROM order_items
    ORDER BY id
    LIMIT 10
    """
)

## Join across tables

In [ ]:
query(
    """
    SELECT
      o.id AS order_id,
      c.email,
      o.status,
      SUM(oi.quantity * oi.unit_price_cents) AS total_cents
    FROM orders o
    JOIN customers c ON c.id = o.customer_id
    JOIN order_items oi ON oi.order_id = o.id
    WHERE o.status = 'pending'
    GROUP BY o.id, c.email, o.status
    ORDER BY o.created_at DESC
    LIMIT 50
    """
)